# 0. Imports

In [149]:
import re
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, Dataset

# 1. Download and Store Dataset

In [150]:
def ingest_reddit_data(subreddit_key: str, n_rows: int = 1_000_000, force_rerun: bool = False) -> Path:
    """
    Orchestrates the ETL process for a specific subreddit's comment data.
    
    Args:
        subreddit_key: Dictionary key from 'splits' (e.g., 'changemyview').
        n_rows: Maximum records to process for the local sample.
        force_rerun: If True, bypasses existence check and overwrites existing parquet file.
        
    Returns:
        Path to the processed Parquet file.
    """
    out_path = Path(f"data/processed/{subreddit_key}_sample.parquet")
    
    # Idempotency check: Skip heavy network I/O if the target file is already present
    if out_path.exists() and not force_rerun:
        print(f"Skipping ingestion: Local cache found at {out_path}")
        return out_path

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Define schema subset based on downstream analytical requirements
    feature_cols = [
        "author", "body", "created_utc", "id", "link_id", "name",
        "parent_id", "score", "controversiality", "total_awards_received"
    ]
    splits = {
    'changemyview': 'data/changemyview-*-of-*.parquet',
    }

    print(f"Streaming data from HuggingFace for: r/{subreddit_key}...")
    
    # Execute lazy-evaluated ETL pipeline
    try:
        (
            pl.scan_parquet(f"hf://datasets/HuggingFaceGECLM/REDDIT_comments/{splits[subreddit_key]}")
            .select(feature_cols)
            # Filter out deleted/removed content to maintain high data quality for NLP tasks
            .filter(~pl.col("body").is_in(["[deleted]", "[removed]"]))
            .limit(n_rows)
            # Stream directly to disk using ZSTD to balance compression ratio and write speed
            .sink_parquet(out_path, compression="zstd")
        )
        print(f"Successfully wrote {n_rows} rows to {out_path}")
    except KeyError:
        raise ValueError(f"Subreddit '{subreddit_key}' not found in defined splits.")
    except Exception as e:
        print(f"Pipeline failed: {e}")
        raise

    return out_path

# --- Execution Control ---
# Toggle 'force_rerun' if the upstream data schema changes or a larger sample is needed
OUT = ingest_reddit_data("changemyview", n_rows=1_000_000, force_rerun=False)

Skipping ingestion: Local cache found at data/processed/changemyview_sample.parquet


# 2. Load Data from Parquet File

In [151]:
df = (
    # Scan the metadata and define the lazy query plan
    pl.scan_parquet("data/processed/changemyview_clean_head.parquet")
      # Constrain sample size for rapid local prototyping
      .head(20000)
      # Trigger execution and load into memory
      .collect()
      # Bridge to Pandas for ecosystem compatibility
      .to_pandas()
)

# 3. Create Train and Test Dataset

### 3.1 Data Preprocessing, Temporal Splitting & Metadata Mapping

In [152]:
# --- 1. Data Cleaning & Type Casting ---

# Ensure text integrity by removing null observations in the primary feature
df = df.dropna(subset=["body"])

# Filter out anonymous/deleted accounts to maintain attribution quality
df = df[df["author"] != "[deleted]"]

# Normalize timestamps: Convert raw strings to numeric Unix seconds, then to datetime objects
# 'coerce' handles malformed strings by returning NaT, preventing pipeline crashes
df["created_utc"] = pd.to_numeric(df["created_utc"], errors="coerce")
df["date"] = pd.to_datetime(df["created_utc"], unit="s", errors="coerce")


# --- 2. Temporal Train/Test Split ---

# Use a temporal 80/20 split rather than a random shuffle to prevent 'look-ahead' bias.
# This simulates a real-world scenario where we predict future comments based on past data.
cutoff = df["created_utc"].quantile(0.8)

df_train = df[df["created_utc"] <= cutoff].copy()
df_test  = df[df["created_utc"] > cutoff].copy()


# --- 3. Metadata Mapping (Lookup Tables) ---

# Create lightweight author lookups for efficient O(1) retrieval.
# Mappings are scoped strictly within splits to enforce isolation and prevent leakage.
id2author_train = df_train.set_index("id")["author"].to_dict()
id2author_test  = df_test.set_index("id")["author"].to_dict()

### 3.2 Interaction Network Construction

In [153]:
def build_reply_pairs(df_split, id2author):
    """
    Constructs a positive interaction dataset by mapping comments to their parent authors.
    Filters for comment-to-comment replies and removes self-interactions.
    """
    # Reddit 'parent_id' prefixes: t1 = Comment, t3 = Link/Post.
    # We restrict analysis to comment-to-comment interactions to capture conversational dynamics.
    parent_comment_ids = df_split["parent_id"].astype(str)
    is_comment_reply = parent_comment_ids.str.startswith("t1_")
    df_r = df_split[is_comment_reply].copy()

    # Extract the raw 36-base ID by stripping the 't1_' type prefix for join compatibility
    df_r["parent_key"] = df_r["parent_id"].str.replace("^t1_", "", regex=True)

    # Resolve parent author identities via the provided lookup table (O(1) mapping)
    df_r["parent_author"] = df_r["parent_key"].map(id2author)

    # --- Data Integrity & Quality Filtering ---
    # 1. Drop replies where the parent comment falls outside the current split (boundary integrity)
    df_r = df_r.dropna(subset=["parent_author"])
    # 2. Exclude self-replies to ensure we only model interpersonal interactions
    df_r = df_r[df_r["author"] != df_r["parent_author"]]

    # Feature selection and renaming to standard (u, v) graph notation
    pairs_pos = df_r[["author", "parent_author", "created_utc", "link_id", "id", "parent_key"]].copy()
    pairs_pos = pairs_pos.rename(columns={
        "author": "u", 
        "parent_author": "v", 
        "id": "u_comment_id", 
        "parent_key": "v_comment_id"
    })
    
    # Label as positive instances for downstream binary classification
    pairs_pos["y"] = 1
    return pairs_pos

# Generate interaction sets; scoped within splits to prevent data leakage
pos_train = build_reply_pairs(df_train, id2author_train)
pos_test  = build_reply_pairs(df_test, id2author_test)

print(f"Positive samples - Train: {len(pos_train):,} | Test: {len(pos_test):,}")

Positive samples - Train: 8,654 | Test: 1,799


In [154]:
# --- Graph Diagnostics: Sparsity & Degree Distribution ---

# Calculate the ratio of users who engaged in at least one reply
pos_users = set(pos_train["u"]) | set(pos_train["v"])
all_users = set(df_train["author"].dropna().unique())
print(f"Engagement Coverage: {len(pos_users)} / {len(all_users)} users with interactions")

# Analyze the 'Out-Degree' (number of replies sent per user)
print("\nReplies per user statistics:")
print(pos_train.groupby("u").size().describe())

Engagement Coverage: 1666 / 2230 users with interactions

Replies per user statistics:
count    1486.000000
mean        5.823688
std        16.001469
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       387.000000
dtype: float64


### 3.3 Negative Sampling Strategy

In [155]:
def build_hard_negatives(df_split, pos_pairs, k_per_pos=2, seed=42):
    """
    Generates 'hard' negative samples for link prediction by identifying potential 
    interactions that did NOT occur within the same discussion thread context.
    """
    # Initialize a BitGenerator for reproducible stochastic sampling
    rng = np.random.default_rng(seed)

    # 1) Contextual Mapping: Identify all active participants per discussion thread (link_id).
    # This defines our 'closed-world' candidate pool for each observation.
    thread_users = (
        df_split.groupby("link_id")["author"]
        .apply(lambda s: set(s.dropna()))
        .to_dict()
    )

    # 2) Network Topology: Extract existing interaction edges in (u, v) space.
    # We treat edges as symmetric to prevent sampling reciprocal replies as negatives,
    # which would introduce label noise.
    reply_edges = set(zip(pos_pairs["u"], pos_pairs["v"]))
    reply_edges_sym = reply_edges | {(v, u) for (u, v) in reply_edges}

    neg_rows = []
    # Project to minimal feature set to reduce overhead during iteration
    pos_pairs_small = pos_pairs[["u", "v", "link_id"]].copy()

    for u, v, link_id in pos_pairs_small.itertuples(index=False):
        users = list(thread_users.get(link_id, []))
        if len(users) <= 1:
            continue

        # Candidate Filtering: 
        # Target users in the same thread (high-signal 'hard' negatives) excluding the source 'u'
        cand = [x for x in users if x != u]
        if not cand:
            continue

        # Collision Avoidance: Remove candidates where a ground-truth interaction (u, x) exists
        cand = [x for x in cand if (u, x) not in reply_edges_sym]
        if not cand:
            continue

        # Stochastic Sampling: Select 'k' negatives per positive to maintain class ratio
        take = min(k_per_pos, len(cand))
        sampled = rng.choice(cand, size=take, replace=False)

        for x in sampled:
            neg_rows.append((u, x, link_id, 0))

    return pd.DataFrame(neg_rows, columns=["u", "v", "link_id", "y"])

# --- Dataset Assembly ---

# Generate split-specific negatives to ensure no data leakage across the temporal boundary
neg_train = build_hard_negatives(df_train, pos_train, k_per_pos=2)
neg_test  = build_hard_negatives(df_test,  pos_test,  k_per_pos=2)

# Final Concatenation & Shuffling:
# We combine positive (y=1) and negative (y=0) instances, then shuffle to ensure 
# gradient descent isn't biased by class-ordered mini-batches.
train_pairs = pd.concat([pos_train[["u","v","link_id","y"]], neg_train], ignore_index=True).sample(frac=1, random_state=42)
test_pairs  = pd.concat([pos_test[["u","v","link_id","y"]],   neg_test],  ignore_index=True).sample(frac=1, random_state=42)

# Diagnostic: Verify class balance (standard ratio is 1:k)
print("Training Class Distribution:\n", train_pairs["y"].value_counts())
print("Testing Class Distribution:\n", test_pairs["y"].value_counts())

Training Class Distribution:
 y
0    16793
1     8654
Name: count, dtype: int64
Testing Class Distribution:
 y
0    3384
1    1799
Name: count, dtype: int64


In [156]:
# --- Unit Tests: Sampling Integrity ---

# Ensure no user is paired with themselves (self-loops)
assert (train_pairs["u"] != train_pairs["v"]).all(), "Found self-interactions in training set"

# Verify that negative samples do not overlap with ground-truth positive replies
real_edges = set(zip(pos_train["u"], pos_train["v"]))
assert not any(
    (u, v) in real_edges or (v, u) in real_edges
    for u, v in zip(neg_train["u"], neg_train["v"])
), "Negative samples contain real interaction edges"

### 3.4 User Textual Profile Construction

In [157]:
# --- 1. Corpus Preparation & Leakage Prevention ---

# Isolate training and testing text to ensure that future comments do not 
# influence the historical representations of users in the training set.
df_train_text = df_train.dropna(subset=["body", "id"]).copy()
df_test_text = df_test.dropna(subset=["body", "id"]).copy()

# (Optional) Heuristic: Filter for active users to ensure embeddings have 
# sufficient signal (min 5 observations). 
# df_train_text = df_train_text.groupby("id").filter(lambda g: len(g) >= 5)

# --- 2. Temporal Aggregation (Feature Engineering) ---

# Construct a profile for each author.
# We join the most recent comments to capture the user's current interests/voice.
user_text_train = (
    df_train_text
    .sort_values("created_utc")             # Enforce chronology to correctly identify the 'tail'
    .groupby("author")["body"]
    # Hyperparameter: Concatenating the last 10 comments balances context vs. sequence length
    .apply(lambda s: " ".join(s.tail(10)))  
)

user_text_test = (
    df_test_text
    .sort_values("created_utc")
    .groupby("author")["body"]
    .apply(lambda s: " ".join(s.tail(10)))
)

# Convert to hash maps (dict) for O(1) lookup performance during the mapping phase
user_text_dict_train = user_text_train.to_dict()
user_text_dict_test = user_text_test.to_dict()

# --- 3. Coverage Analysis (Data Integrity Check) ---

# Quantify the 'Cold-Start' issue: users in the interaction pairs who lack 
# textual history. Significant missingness here indicates a sampling mismatch.
missing_train = train_pairs["u"].map(user_text_dict_train).isna().mean()
print(f"Missing text profile ratio (Train - Source User): {missing_train:.2%}")

missing_test = test_pairs["u"].map(user_text_dict_test).isna().mean()
print(f"Missing text profile ratio (Test - Source User): {missing_test:.2%}")

Missing text profile ratio (Train - Source User): 0.00%
Missing text profile ratio (Test - Source User): 0.00%


In [158]:
# --- 1. Prepare Data Containers ---

# Create copies to prevent SettingWithCopy warnings and isolate split changes
train_pairs = train_pairs.copy()
test_pairs  = test_pairs.copy()

def attach_text(pairs, user_text_dict):
    """Adds historical text for source (u) and target (v) users."""
    
    # Map text profiles to user IDs
    pairs["text_u"] = pairs["u"].map(user_text_dict)
    pairs["text_v"] = pairs["v"].map(user_text_dict)
    
    # Remove observations missing text for either user to ensure a complete feature set
    return pairs.dropna(subset=["text_u", "text_v"])

# --- 2. Execute Merge & Cleanup ---

train_pairs_txt = attach_text(train_pairs, user_text_dict_train)
test_pairs_txt  = attach_text(test_pairs,  user_text_dict_test)

# --- 3. Progress Check ---

# Log row counts to monitor data loss during the mapping/dropping process
print(f"Train Retention: {len(train_pairs):,} -> {len(train_pairs_txt):,}")
print(f"Test Retention:  {len(test_pairs):,} -> {len(test_pairs_txt):,}")

Train Retention: 25,447 -> 25,447
Test Retention:  5,183 -> 5,183


### 3.5 Save Train and Test Datasets

In [159]:
train_pairs_txt.to_parquet("data/processed/train_pairs_txt.parquet", index=False)
test_pairs_txt.to_parquet("data/processed/test_pairs_txt.parquet", index=False)

# 4. Train CNN

### 4.1 Create Vocabulary

In [160]:
# Define a simple regex to extract word-level tokens (alphabetic only)
TOKEN_RE = re.compile(r"[A-Za-z']+")

def tokenize(text: str):
    """Lowercases and extracts valid word tokens from raw text."""
    return TOKEN_RE.findall(text.lower())

# Constraints for memory efficiency and noise reduction
MAX_VOCAB = 50_000
MIN_FREQ = 2

# Build frequency distribution from training corpus only to prevent leakage
counter = Counter()
for t in train_pairs_txt["text_u"].tolist():
    counter.update(tokenize(t))
for t in train_pairs_txt["text_v"].tolist():
    counter.update(tokenize(t))

# Reserved tokens for sequence padding and out-of-vocabulary terms
PAD = "<pad>"
UNK = "<unk>"

# Initialize vocabulary with reserved indices
vocab = {PAD: 0, UNK: 1}

# Populate vocabulary with the most frequent terms meeting the frequency threshold
for w, c in counter.most_common(MAX_VOCAB):
    if c < MIN_FREQ:
        break
    vocab[w] = len(vocab)

pad_id = vocab[PAD]
unk_id = vocab[UNK]

print(f"Final Vocab Size: {len(vocab):,}")

Final Vocab Size: 25,626


### 4.2 Dataset Definition & User Text Encoding

In [161]:
# Fixed sequence length to ensure uniform input dimensions for the model
MAX_LEN = 256  

def encode(text: str):
    """
    Converts raw text into a list of integer IDs.
    Unknown words are mapped to 'unk_id' and sequences are truncated to MAX_LEN.
    """
    ids = [vocab.get(w, unk_id) for w in tokenize(text)]
    return ids[:MAX_LEN]

class PairDataset(Dataset):
    """
    Custom PyTorch Dataset to serve (u, v) pairs, their respective 
    encoded histories, and the binary interaction label.
    """
    def __init__(self, df_pairs):
        self.u = df_pairs["u"].tolist()
        self.v = df_pairs["v"].tolist()
        self.u_texts = df_pairs["text_u"].tolist()
        self.v_texts = df_pairs["text_v"].tolist()
        self.y = df_pairs["y"].astype(float).tolist()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Returns raw IDs and encoded text sequences for the given index
        return (
            self.u[idx],
            self.v[idx],
            encode(self.u_texts[idx]),
            encode(self.v_texts[idx]),
            self.y[idx],
        )

In [162]:
def collate_fn(batch):
    """
    Dynamic padding: Aligns sequences within a batch to the length of 
    the longest sequence found in that specific batch.
    """
    # Unpack columns from the batch of tuples
    u, v, u_seqs, v_seqs, ys = zip(*batch)

    # Track original lengths for masking or sequence packing
    u_lens = torch.tensor([len(s) for s in u_seqs], dtype=torch.long)
    v_lens = torch.tensor([len(s) for s in v_seqs], dtype=torch.long)

    # Determine batch-wide maximum dimensions
    max_u = max(u_lens).item()
    max_v = max(v_lens).item()

    # Initialize tensors filled with the PAD token
    u_tensor = torch.full((len(batch), max_u), pad_id, dtype=torch.long)
    v_tensor = torch.full((len(batch), max_v), pad_id, dtype=torch.long)

    # Copy sequence data into the padded containers
    for i, s in enumerate(u_seqs):
        u_tensor[i, :len(s)] = torch.tensor(s, dtype=torch.long) 
        
    for i, s in enumerate(v_seqs):
        v_tensor[i, :len(s)] = torch.tensor(s, dtype=torch.long) 

    # Convert labels to standard floating-point tensor
    y = torch.tensor(ys, dtype=torch.float32)

    return list(u), list(v), u_tensor, v_tensor, u_lens, v_lens, y

### 4.3 Data Loader Initilization

In [163]:
# Number of samples processed before the model updates its internal parameters
BATCH_SIZE = 128

# Instantiate dataset objects for training and evaluation
train_ds = PairDataset(train_pairs_txt)
test_ds  = PairDataset(test_pairs_txt)

# Training Loader: Shuffle enabled to prevent the model from learning the order of samples
train_loader = DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn
)

# Testing Loader: Shuffle disabled to ensure consistent, reproducible evaluation
test_loader = DataLoader(
    test_ds,  
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
)

### 4.4 Create Siamese CNN

##### 4.4.1 Create Text Encoder to build user embeddings

In [164]:
class TextCNNEncoder(nn.Module):
    """
    Multi-kernel CNN for extracting hierarchical n-gram features from text.
    Outputs a normalized embedding representing a user's linguistic style.
    """
    def __init__(self, vocab_size, emb_dim=128, num_filters=128, kernel_sizes=(3,4,5), out_dim=128, pad_idx=0, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        
        # Parallel convolutional layers to capture varied phrase lengths
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=emb_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), out_dim)

    def forward(self, x):
        # x: [Batch, Sequence_Length]
        emb = self.embedding(x)            
        emb = emb.transpose(1, 2)          # Align dimensions for 1D convolution [B, E, T]

        conv_outs = []
        for conv in self.convs:
            # Apply convolution and non-linearity
            h = F.relu(conv(emb))          
            # Global Max Pooling: Extract the most salient feature per filter
            h = F.max_pool1d(h, kernel_size=h.size(2)).squeeze(2)  
            conv_outs.append(h)

        # Fusion of multi-scale features
        h = torch.cat(conv_outs, dim=1)    
        h = self.dropout(h)
        h = self.fc(h)                     
        
        # L2 Normalization to facilitate cosine similarity downstream
        h = F.normalize(h, p=2, dim=1)     
        return h

##### 4.4.2 Create Siamese Architecture

In [165]:
class SiameseCNN(nn.Module):
    """
    Siamese architecture for metric learning. 
    Uses a shared encoder to map both users to a common vector space, 
    measuring interaction probability via scaled cosine similarity.
    """
    def __init__(self, encoder: nn.Module, scale=10.0):
        super().__init__()
        # Shared weight mechanism: Both inputs use the exact same feature extractor
        self.encoder = encoder
        
        # Temperature scaling factor to expand cosine range [-1, 1] 
        # into logits capable of driving sigmoid probabilities near 0 or 1.
        self.scale = scale

    def forward(self, u_tensor, v_tensor):
        # 1. Feature Extraction (Shared Weights)
        eu = self.encoder(u_tensor)
        ev = self.encoder(v_tensor)
        
        # 2. Similarity Metric (Dot Product on Unit Vectors = Cosine Similarity)
        # Result shape: [Batch_Size]
        cos = (eu * ev).sum(dim=1)         
        
        # 3. Logit Scaling
        # Prepares the values for the Binary Cross Entropy loss function
        logits = self.scale * cos          

        return logits

##### 4.4.3 Model Instatntiation & Device Allocation

In [166]:
# Automatically detect if a GPU is available for accelerated training
device = torch.device("mps" if torch.mps.is_available() else "cpu")

# Initialize the Feature Extractor (Encoder)
# We use a Multi-Kernel CNN to capture n-gram patterns of lengths 3, 4, and 5
encoder = TextCNNEncoder(
    vocab_size=len(vocab),  # Determined by the tokenizer in Section 6
    emb_dim=128,            # Dimensionality of the dense word vectors
    num_filters=128,        # Number of features to extract per kernel size
    kernel_sizes=(3, 4, 5), # Window sizes: Tri-grams, 4-grams, 5-grams
    out_dim=128,            # Final embedding size (compact semantic vector)
    pad_idx=pad_id,         # Index to ignore during embedding lookup (zero gradient)
    dropout=0.2,            # Regularization to prevent overfitting on specific phrases
)

# Wrap the encoder in the Siamese architecture for metric learning
# scale=10.0 expands the cosine range [-1, 1] to [-10, 10] for sharper probability gradients
model = SiameseCNN(encoder, scale=10.0).to(device)

print(f"Model initialized on: {device}")

Model initialized on: mps


### 4.5 Train the CNN and Evaluate

In [167]:
# --- Corrected Training & Evaluation Loop ---

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)

@torch.no_grad()
def eval_auc(model, loader):
    model.eval()
    all_targets = []
    all_probs = []
    
    # 1. Loop through data
    # Note: We use *ignore syntax to handle the extra items (IDs, lengths) safely
    # collate_fn returns 7 items: (u_ids, v_ids, u_tensor, v_tensor, u_lens, v_lens, y)
    for _, _, u_tensor, v_tensor, _, _, y in loader:
        
        # Move tensors to GPU
        u_tensor = u_tensor.to(device)
        v_tensor = v_tensor.to(device)
        
        # Get predictions
        logits = model(u_tensor, v_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()
        
        # Store results
        all_targets.extend(y.numpy())
        all_probs.extend(probs)

    # 2. One-line AUC calculation
    # Handle the edge case where you only have 0s or only 1s (AUC is undefined)
    try:
        return roc_auc_score(all_targets, all_probs)
    except ValueError:
        return float("nan")

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    
    # CORRECT UNPACKING for Training
    for _, _, u_tensor, v_tensor, _, _, y in loader:
        
        # Send text tensors to GPU
        u_tensor, v_tensor, y = u_tensor.to(device), v_tensor.to(device), y.to(device)

        optimizer.zero_grad(set_to_none=True)
        
        logits = model(u_tensor, v_tensor)
        
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * len(y)
        
    return total_loss / len(loader.dataset)

# --- Run ---
EPOCHS = 5
for epoch in range(1, EPOCHS + 1):
    loss = train_one_epoch(model, train_loader)
    auc = eval_auc(model, test_loader)
    train_auc = eval_auc(model, train_loader)
    print(f"Train AUC: {train_auc}")
    print(f"epoch={epoch}  loss={loss:.4f}  test_auc={auc:.4f}")

Train AUC: 0.7465060634244978
epoch=1  loss=1.0243  test_auc=0.5554
Train AUC: 0.8674827623805913
epoch=2  loss=0.7892  test_auc=0.5370
Train AUC: 0.8940255729607478
epoch=3  loss=0.6996  test_auc=0.5376
Train AUC: 0.9361551973595037
epoch=4  loss=0.6272  test_auc=0.5500
Train AUC: 0.9568056223036685
epoch=5  loss=0.5688  test_auc=0.5357


In [168]:
@torch.no_grad()
def save_predictions_parquet(model, loader, out_path: str):
    model.eval()
    rows = []

    # Unpack 7 items (u_ids, v_ids, u_tensor, v_tensor, u_lens, v_lens, y)
    for u_ids, v_ids, u_tensor, v_tensor, _, _, y in loader:
        
        # Move tensors to GPU for calculation
        u_tensor = u_tensor.to(device)
        v_tensor = v_tensor.to(device)
        
        # We generally DO NOT move targets 'y' to GPU during inference 
        # unless we need to calculate loss. If you did move it, you must move it back.
        
        logits = model(u_tensor, v_tensor)
        probs = torch.sigmoid(logits).cpu().numpy()

        # --- FIX: Ensure y is on CPU before converting ---
        y_np = y.cpu().numpy()

        for u, v, yt, sc in zip(u_ids, v_ids, y_np, probs):
            rows.append({
                "u": u,
                "v": v,
                "y": int(yt),
                "score": float(sc),
            })

    df_out = pd.DataFrame(rows)
    df_out.to_parquet(out_path, index=False)
    return df_out

pred_df = save_predictions_parquet(model, test_loader, "predictions_test.parquet")
print(pred_df.head())
print("Saved:", len(pred_df), "rows")

                    u            v  y     score
0        Joined_Today     mcbarron  0  0.087121
1          bluntzfang   selementar  1  0.662098
2  queen_of_spades513  ruat_caelum  1  0.865705
3            mcbarron    starfirex  0  0.336061
4    mario_meowingham  phoenixrawr  0  0.603637
Saved: 5183 rows
